In [8]:
import subprocess
subprocess.run(["pip", "install", "psycopg2-binary", "sqlalchemy"], check=True)
print("Dependencies installed!")

Dependencies installed!


# 04 — Orders Transformation (PySpark)
Reads raw.orders from PostgreSQL, validates and cleans the data using PySpark,
then loads clean records into staging.orders.
PySpark is used here because orders is the largest and fastest growing collection.

In [9]:
import urllib.request
import os

# Download to home directory — jovyan user always has write access here
jar_path = "/home/jovyan/postgresql-42.7.3.jar"

if not os.path.exists(jar_path):
    print("Downloading PostgreSQL JDBC driver...")
    urllib.request.urlretrieve(
        "https://jdbc.postgresql.org/download/postgresql-42.7.3.jar",
        jar_path
    )
    print("Downloaded!")
else:
    print("JDBC driver already exists — skipping download.")

print(f"JAR path: {jar_path}")

JDBC driver already exists — skipping download.
JAR path: /home/jovyan/postgresql-42.7.3.jar


In [10]:
# Stop any existing Spark session so we can create a fresh one
# with the JDBC driver properly configured
try:
    spark.stop()
    print("Existing Spark session stopped.")
except:
    print("No existing session to stop.")

Existing Spark session stopped.


In [11]:
from pyspark.sql import SparkSession

jar_path = "/home/jovyan/postgresql-42.7.3.jar"

spark = SparkSession.builder \
    .appName("MANDERA_ANALYTICS — Orders Transform") \
    .config("spark.jars", jar_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print("Spark session started!")

Spark version: 3.5.0
Spark session started!


## Step 1 — Load raw orders from PostgreSQL

In [12]:
import os

JDBC_URL = (
    f"jdbc:postgresql://{os.environ['POSTGRES_HOST']}:"
    f"{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

JDBC_PROPERTIES = {
    "user": os.environ["POSTGRES_USER"],
    "password": os.environ["POSTGRES_PASSWORD"],
    "driver": "org.postgresql.Driver",
}

# Read raw.orders into a Spark DataFrame via JDBC
df = spark.read.jdbc(
    url=JDBC_URL,
    table="raw.orders",
    properties=JDBC_PROPERTIES,
)

print(f"Loaded {df.count():,} raw order records")
print(f"\nSchema:")
df.printSchema()

Loaded 37,891 raw order records

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: short (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- total_price: decimal(10,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_method: string (nullable = true)
 |-- batch_id: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)



## Step 2 — Validate

In [13]:
from pyspark.sql.functions import col, count, when, isnan

print(" NULL CHECKS ")
required_fields = ["order_id", "customer_id", "product_id", "quantity", "unit_price"]

for field in required_fields:
    null_count = df.filter(col(field).isNull()).count()
    status = "PASS" if null_count == 0 else "FAIL"
    print(f"  [{status}] {field}: {null_count} nulls")

print("\n VALUE CHECKS ")
invalid_qty   = df.filter(col("quantity") <= 0).count()
invalid_price = df.filter(col("unit_price") <= 0).count()
duplicates    = df.count() - df.dropDuplicates(["order_id"]).count()

print(f"  [{'PASS' if invalid_qty == 0 else 'FAIL'}] quantity <= 0: {invalid_qty}")
print(f"  [{'PASS' if invalid_price == 0 else 'FAIL'}] unit_price <= 0: {invalid_price}")
print(f"  [{'PASS' if duplicates == 0 else 'FAIL'}] order_id duplicates: {duplicates}")

print("\n STATUS VALUES ")
df.groupBy("status").count().orderBy("count", ascending=False).show()

 NULL CHECKS 
  [PASS] order_id: 0 nulls
  [PASS] customer_id: 0 nulls
  [PASS] product_id: 0 nulls
  [PASS] quantity: 0 nulls
  [PASS] unit_price: 0 nulls

 VALUE CHECKS 
  [PASS] quantity <= 0: 0
  [PASS] unit_price <= 0: 0
  [PASS] order_id duplicates: 0

 STATUS VALUES 
+---------+-----+
|   status|count|
+---------+-----+
|  shipped| 7669|
|delivered| 7650|
| returned| 7544|
|  pending| 7519|
|cancelled| 7509|
+---------+-----+



## Step 3 — Clean and Transform

In [14]:
from pyspark.sql.functions import lower, trim, round as spark_round, col, current_timestamp

df_clean = df \
    .withColumn("status",          lower(trim(col("status")))) \
    .withColumn("payment_method",  lower(trim(col("payment_method")))) \
    .withColumn("shipping_method", lower(trim(col("shipping_method")))) \
    .withColumn("total_price",     spark_round(col("quantity") * col("unit_price"), 2)) \
    .withColumn("loaded_at",       current_timestamp())

print(f"Transformed {df_clean.count():,} order records")
df_clean.show(5)

Transformed 37,891 order records
+--------+-----------+----------+--------+----------+-----------+---------+--------------+---------------+--------+--------------------+--------------------+
|order_id|customer_id|product_id|quantity|unit_price|total_price|   status|payment_method|shipping_method|batch_id|          created_at|           loaded_at|
+--------+-----------+----------+--------+----------+-----------+---------+--------------+---------------+--------+--------------------+--------------------+
|ORD55633|  cust85698|  PROD4207|       7|    306.69|    2146.83|  pending| bank_transfer|         pickup|       4|2026-04-18 03:37:...|2026-04-22 18:12:...|
|ORD75449|  cust26604|  PROD7597|       7|    425.53|    2978.71| returned|   credit_card|        express|       4|2026-04-18 03:37:...|2026-04-22 18:12:...|
|ORD67926|  cust46901|  PROD9311|       1|    269.97|     269.97|delivered| bank_transfer|        express|       4|2026-04-18 03:37:...|2026-04-22 18:12:...|
|ORD37896|  cust636

## Step 4 — Create staging table and load

In [15]:
from sqlalchemy import create_engine, text
import os

engine = create_engine(
    f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS staging.orders (
            order_id        VARCHAR PRIMARY KEY,
            customer_id     VARCHAR NOT NULL,
            product_id      VARCHAR NOT NULL,
            quantity        SMALLINT NOT NULL,
            unit_price      NUMERIC(10, 2) NOT NULL,
            total_price     NUMERIC(10, 2) NOT NULL,
            status          VARCHAR NOT NULL,
            payment_method  VARCHAR,
            shipping_method VARCHAR,
            batch_id        INTEGER,
            created_at      TIMESTAMP WITH TIME ZONE,
            loaded_at       TIMESTAMP WITH TIME ZONE
        );
    """))
    conn.commit()

print("staging.orders table ready!")

staging.orders table ready!


In [16]:
# Write clean orders to staging using PySpark JDBC writer
df_clean.write.jdbc(
    url=JDBC_URL,
    table="staging.orders",
    mode="append",
    properties=JDBC_PROPERTIES,
)

print(f"Loaded {df_clean.count():,} orders into staging.orders")

Loaded 37,891 orders into staging.orders


In [17]:
import pandas as pd

df_verify = pd.read_sql("SELECT COUNT(*) as row_count FROM staging.orders;", engine)
print(f"staging.orders row count: {df_verify['row_count'][0]:,}")

staging.orders row count: 37,891
